In [ ]:
import kagglehub
import os
import shutil

In [ ]:
def download_kaggle_file(file_name: str):

    # Clear cache
    cache_dir = os.path.expanduser("~/.cache/kagglehub")

    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir)
        print("KaggleHub cache cleared.")
    else:
        print("No cache found.")

    # 1. Download
    temp_path = kagglehub.dataset_download(file_name)
    
    # 2. Set new directory
    current_dir = os.getcwd()
    data_dir = current_dir + "/data/source_data"
    
    # 3. Move files
    print(f"Moving files from {temp_path} to {data_dir}...")
    
    files_moved = 0
    for filename in os.listdir(temp_path):
        source = os.path.join(temp_path, filename)
        destination = os.path.join(data_dir, filename)
        
        # Check if it's a file (skip subdirectories if any)
        if os.path.isfile(source):
            shutil.move(source, destination)
            print(f"Moved: {filename}")
            files_moved += 1
            
    print(f"\nFinished. {files_moved} files are now in source data folder")

In [ ]:
download_kaggle_file("logandonaldson/sports-stadium-locations")

In [ ]:
import pandas as pd
import geopandas as gpd

In [ ]:
pro_stadiums = pd.read_csv("data/source_data/stadiums.csv")

In [ ]:
pro_stadiums.head()

In [ ]:
nba_stadiums = gpd.GeoDataFrame(
    pro_stadiums[pro_stadiums["League"] == "NBA"],
    geometry=gpd.points_from_xy(pro_stadiums[pro_stadiums["League"] == "NBA"]['Long'], pro_stadiums[pro_stadiums["League"] == "NBA"]['Lat']),
    crs="EPSG:4326")[["Team", "geometry"]]

In [ ]:
nba_stadiums

In [ ]:
nba_stadiums.loc[nba_stadiums['Team'] == 'Los Angeles Clippers', 'Team'] = "LA Clippers"
nba_stadiums.loc[nba_stadiums['Team'] == 'Sacremento Kings', 'Team'] = "Sacramento Kings"

In [ ]:
import time
from nba_api.stats.endpoints import leaguestandings

In [ ]:
def get_nba_win_history(years=5):
    # Determine the current year for the 2025-26 season
    current_year = 2025
    all_seasons_data = []

    for i in range(years):
        # Format season string as 'YYYY-YY' (e.g., '2024-25')
        year_start = current_year - i
        year_end = str(year_start + 1)[2:]
        season_str = f"{year_start}-{year_end}"
        
        print(f"Fetching standings for {season_str}...")
        
        try:
            # Call the league standings endpoint
            standings = leaguestandings.LeagueStandings(season=season_str)
            df = standings.get_data_frames()[0]

            # Extract relevant columns
            season_df = df[['TeamCity', 'TeamName', 'WINS', 'LOSSES']].copy()
            season_df['Season'] = season_str
            season_df['Team'] = df['TeamCity'] + ' ' + df['TeamName']
            all_seasons_data.append(season_df)
            
            # Brief sleep to avoid hitting NBA.com rate limits
            time.sleep(1)
            
        except Exception as e:
            print(f"Could not retrieve data for {season_str}: {e}")

    # Combine all seasons into one DataFrame
    final_df = pd.concat(all_seasons_data, ignore_index=True)
    
    return final_df

In [ ]:
win_pct_history = get_nba_win_history(5)

In [ ]:
team_success = (
    win_pct_history.groupby("Team")
      .agg(
          total_wins=("WINS", "sum"),
          total_losses=("LOSSES", "sum")
      )
)

# Add win percentage
team_success["win_pct"] = team_success["total_wins"] / (team_success["total_wins"] + team_success["total_losses"])

team_success = team_success.reset_index(names='Team')

In [ ]:
team_success

In [ ]:
import requests
from io import StringIO

In [ ]:
def get_nba_valuation(url: str):
    # Spoof a browser User-Agent to bypass the 403
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0 Safari/537.36"
        )
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    # Parse tables from the response text
    tables = pd.read_html(StringIO(response.text))
    print(f"Found {len(tables)} table(s)")

    # Inspect and pick the right table
    for i, df in enumerate(tables):
        print(f"\n--- Table {i} ({df.shape[0]} rows × {df.shape[1]} cols) ---")
        print(df.head(3))

    # Use table 0 (adjust index if needed)
    df = tables[0]

    # Flatten multi-level headers if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [" ".join(str(c) for c in col).strip() for col in df.columns]

    return df

In [ ]:
url = "https://en.wikipedia.org/wiki/Forbes_list_of_the_most_valuable_NBA_teams"
wiki_nba_valuation = get_nba_valuation(url)

In [ ]:
wiki_nba_valuation

In [ ]:
team_valuation = wiki_nba_valuation[["Team", "Value"]]

In [ ]:
def convert_currency_string(val):
    if pd.isna(val) or not isinstance(val, str):
        return val
    
    # Remove '$', 'billion', 'million', and any extra whitespace
    clean_val = val.replace('$', '').replace(' billion', '').strip()
    
    try:
        # Convert to float
        number = float(clean_val)
        
        # Scale based on the suffix
        if 'billion' in val.lower():
            return number * 1_000_000_000
        elif 'million' in val.lower():
            return number * 1_000_000
        return number
    except ValueError:
        return None

In [ ]:
team_valuation['Value'] = team_valuation['Value'].apply(convert_currency_string)

In [ ]:
team_valuation.loc[team_valuation['Team'] == 'Los Angeles Clippers', 'Team'] = "LA Clippers"

In [ ]:
team_valuation.head()

In [ ]:
team_full = gpd.GeoDataFrame(team_valuation.merge(team_success[["Team", "win_pct"]], on="Team").merge(nba_stadiums, on="Team"))
team_full.columns = team_full.columns.str.lower()

In [ ]:
team_full.head()

In [ ]:
team_full.explore(
    column='value', 
    cmap='YlGnBu',
    tiles='CartoDB Positron',
    marker_kwds={
        'radius': 10,
        'fill': True
    },
    style_kwds={
        'stroke': True,
        'weight': 1,
        'color': 'white'
    }
)

In [ ]:
team_full.to_csv('data/processed_data/team_node.csv', index=False)